# Liquid Foundation Models · Notebook de clase

Versión de trabajo en castellano con **Don Quijote**, comparación entre **Transformer**, **Liquid** e **híbrido**, guardado del mejor modelo, gráficas de pérdida y generación con temperatura modificable.

## 1. Idea del experimento

Vamos a comparar tres enfoques sobre la misma tarea de predicción del siguiente carácter:

- **MiniTransformer**: solo atención.
- **MiniLiquid**: solo bloques liquid.
- **HybridLiquid**: mezcla de bloques liquid y atención intercalada.

El objetivo no es entrenar un gran modelo, sino **comparar arquitectura, coste y comportamiento**.

## 2. Importaciones

Cargamos bibliotecas para PyTorch, descarga del texto, gráficas y barra de progreso.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import requests
from pathlib import Path
import time
import math
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

## 3. Descarga y preparación del dataset

Usamos **Don Quijote** en texto plano y tokenización a nivel de carácter. Esto simplifica mucho la demo y permite trabajar en castellano.

In [3]:
# ======================================
# 1. Descarga y preparación de Don Quijote
# ======================================
data_path = Path("el_quijote.txt")

if not data_path.exists():
    print("Descargando Don Quijote...")
    url = "https://www.gutenberg.org/files/2000/2000-0.txt"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text = response.text
    start_marker = "*** START OF THE PROJECT GUTENBERG"
    end_marker = "*** END OF THE PROJECT GUTENBERG"
    start_idx = text.find(start_marker) + len(start_marker)
    end_idx = text.find(end_marker)
    text = text[start_idx:end_idx].strip()
    data_path.write_text(text)

text = data_path.read_text()
print(f"Longitud del texto: {len(text):,} caracteres")

# Char-level tokenizer
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

def get_batch(batch_size=64, block_size=128, device='cpu'):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

Longitud del texto: 1,038,397 caracteres


## 4. Bloques básicos

Aquí aparecen los dos bloques clave:

- `AttentionBlock`, que representa el enfoque tipo Transformer.
- `LiquidBlock`, que combina proyecciones, puertas (*gates*), convolución local y residual.

In [4]:
# ======================================
# 2. Bloques (sin cambios)
# ======================================

class AttentionBlock(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
    
    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm(x + attn_out)
        x = self.norm(x + self.ff(x))
        return x


class LiquidBlock(nn.Module):
    def __init__(self, d_model, kernel_size=5):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.in_proj = nn.Linear(d_model, d_model)
        self.gate1 = nn.Linear(d_model, d_model)
        self.conv = nn.Conv1d(d_model, d_model, kernel_size,
                              padding=kernel_size//2, groups=d_model)
        self.gate2 = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )
    
    def forward(self, x):
        residual = x
        x = self.norm1(x)
        z = self.in_proj(x) * torch.sigmoid(self.gate1(x))
        z = z.transpose(1, 2)
        z = self.conv(z)
        z = z.transpose(1, 2)
        z = z * torch.sigmoid(self.gate2(x))
        z = self.out_proj(z)
        x = residual + z
        x = self.norm2(x)
        x = x + self.ff(x)
        return x

## 5. Modelos completos

Construimos tres arquitecturas comparables para el experimento.

In [ ]:
# ======================================
# 3. Modelos
# ======================================

class MiniTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=6):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([AttentionBlock(d_model, n_heads) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embed(x)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        return self.head(x)


class MiniLiquid(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_layers=12, kernel_size=5):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([LiquidBlock(d_model, kernel_size) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embed(x)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        return self.head(x)


class HybridLiquid(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_layers=12, kernel_size=5):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        layers = []
        for i in range(n_layers):
            if i % 3 == 0:
                layers.append(AttentionBlock(d_model, n_heads=4))
            else:
                layers.append(LiquidBlock(d_model, kernel_size))
        self.layers = nn.ModuleList(layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embed(x)
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        return self.head(x)

## 6. Entrenamiento, guardado y generación

La función principal:

- entrena por épocas;
- guarda el mejor modelo;
- dibuja la curva de pérdida;
- genera texto con temperatura configurable.

In [ ]:
# ======================================
# 4. Entrenamiento + guardado del mejor modelo + gráficas + generación con temperatura
# ======================================
def train_and_generate(model, name, epochs=8, batch_size=64, block_size=128, lr=4e-4, 
                       device='cuda', temperature=0.8, save_dir="models"):
    import os
    os.makedirs(save_dir, exist_ok=True)
    
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
    print(f"\n=== Entrenando {name} ===")
    losses = []
    best_loss = float('inf')
    best_epoch = -1
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        steps = 80  # ajusta según tu hardware (más pasos = mejor pero más lento)
        
        for _ in tqdm(range(steps), desc=f"Epoch {epoch+1}/{epochs}"):
            x, y = get_batch(batch_size, block_size, device)
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / steps
        losses.append(avg_loss)
        print(f"Epoch {epoch+1:2d} | Loss: {avg_loss:.4f}")
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_epoch = epoch + 1
            torch.save(model.state_dict(), f"{save_dir}/{name.replace(' ', '_')}_best.pt")
            print(f"  → Mejor modelo guardado (loss {best_loss:.4f})")
    
    # Gráfica de pérdida
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, epochs+1), losses, marker='o', label=name)
    plt.title(f"Curva de pérdida - {name}")
    plt.xlabel("Época")
    plt.ylabel("Pérdida (cross-entropy)")
    plt.grid(True)
    plt.legend()
    plt.show()
    
    # Generación con temperatura
    print(f"\nGenerando texto con temperatura = {temperature}...")
    model.eval()
    with torch.no_grad():
        context = torch.tensor([stoi[' ']], dtype=torch.long, device=device).unsqueeze(0)
        generated = []
        for _ in range(400):
            logits = model(context)
            logits = logits[:, -1, :] / temperature   # aplicar temperatura
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            context = torch.cat([context, next_idx], dim=1)
            generated.append(itos[next_idx.item()])
        
        print(f"\nTexto generado por {name} (temp={temperature}):\n")
        print(''.join(generated))
        print("-" * 90)

## 7. Ejecución del experimento

Aquí puedes dejar los tres modelos o comentar temporalmente algunos para acelerar la clase.

In [ ]:
# ======================================
# 5. Ejecutar comparación
# ======================================
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Dispositivo: {device}")
    
    d_model = 128
    block_size = 128
    save_dir = "saved_models"
    
    models = [
        ("MiniTransformer", MiniTransformer(vocab_size, d_model, n_heads=4, n_layers=6)),
        ("Pure Liquid", MiniLiquid(vocab_size, d_model, n_layers=12)),
        ("Hybrid LFM-style", HybridLiquid(vocab_size, d_model, n_layers=12))
    ]
    
    TEMPERATURA = 0.85   # ¡Cambia este valor! (0.6 = más determinista, 1.2 = más creativo/loco)
    
    for name, model in models:
        train_and_generate(
            model, 
            name, 
            epochs=8, 
            device=device, 
            temperature=TEMPERATURA,
            save_dir=save_dir
        )